# Chapter 3: Assigning Roles (Role Prompting)

- [Lesson](#lesson)
- [Exercises](#exercises)
- [Example Playground](#example-playground)

## Setup

Run the following setup cell to load your API key and establish the `get_completion` helper function.

In [1]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Lesson

Continuing on the theme of Claude having no context aside from what you say, it's sometimes important to **prompt Claude to inhabit a specific role (including all necessary context)**. This is also known as role prompting. The more detail to the role context, the better.

**Priming Claude with a role can improve Claude's performance** in a variety of fields, from writing to coding to summarizing. It's like how humans can sometimes be helped when told to "think like a ______". Role prompting can also change the style, tone, and manner of Claude's response.

**Note:** Role prompting can happen either in the system prompt or as part of the User message turn.

### Examples

In the example below, we see that without role prompting, Claude provides a **straightforward and non-stylized answer** when asked to give a single sentence perspective on skateboarding.

However, when we prime Claude to inhabit the role of a cat, Claude's perspective changes, and thus **Claude's response tone, style, content adapts to the new role**. 

**Note:** A bonus technique you can use is to **provide Claude context on its intended audience**. Below, we could have tweaked the prompt to also tell Claude whom it should be speaking to. "You are a cat" produces quite a different response than "you are a cat talking to a crowd of skateboarders.

Here is the prompt without role prompting in the system prompt:

In [2]:
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT))

Skateboarding is a creative blend of athleticism, artistry, and problem-solving that appeals to people across different ages and skill levels.


Here is the same user question, except with role prompting.

In [5]:
# System prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

I think skateboarding is a cool blend of athleticism, creativity, and freedom that appeals to people who enjoy pushing their physical and artistic limits.


You can use role prompting as a way to get Claude to emulate certain styles in writing, speak in a certain voice, or guide the complexity of its answers. **Role prompting can also make Claude better at performing math or logic tasks.**

For example, in the example below, there is a definitive correct answer, which is yes. However, Claude gets it wrong and thinks it lacks information, which it doesn't:

In [6]:
# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT))

# Analysis

Let me work through the possible cases:

**Case 1: Anne is married**
- Jack (married) is looking at Anne (married) ✗
- Anne (married) is looking at George (unmarried) ✓

**Case 2: Anne is not married**
- Jack (married) is looking at Anne (unmarried) ✓
- Anne (unmarried) is looking at George (unmarried) ✗

In both possible cases, **yes, a married person is looking at an unmarried person**.

Whether Anne is married or not, at least one married person is looking at an unmarried person (either Jack looking at Anne, or Anne looking at George).


Now, what if we **prime Claude to act as a logic bot**? How will that change Claude's answer? 

It turns out that with this new role assignment, Claude gets it right. (Although notably not for all the right reasons)

In [7]:
# System prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

# Logic Analysis

Let me work through this systematically.

**Given facts:**
- Jack is looking at Anne
- Anne is looking at George
- Jack is married
- George is not married
- Anne's marital status is unknown

**Question:** Is a married person looking at an unmarried person?

**Analysis:**

There are two possible cases for Anne:

**Case 1: Anne is married**
- Jack (married) is looking at Anne (married) ✗
- Anne (married) is looking at George (unmarried) ✓

**Case 2: Anne is not married**
- Jack (married) is looking at Anne (unmarried) ✓
- Anne (unmarried) is looking at George (unmarried) ✗

**Answer: YES**

In both possible cases, there is at least one instance of a married person looking at an unmarried person. In Case 1, it's Anne looking at George. In Case 2, it's Jack looking at Anne. Since we must answer whether this occurs, and it's true in all possible scenarios, the answer is **yes**.


**Note:** What you'll learn throughout this course is that there are **many prompt engineering techniques you can use to derive similar results**. Which techniques you use is up to you and your preference! We encourage you to **experiment to find your own prompt engineering style**.

If you would like to experiment with the lesson prompts without changing any content above, scroll all the way to the bottom of the lesson notebook to visit the [**Example Playground**](#example-playground).

---

## Exercises
- [Exercise 3.1 - Math Correction](#exercise-31---math-correction)

### Exercise 3.1 - Math Correction
In some instances, **Claude may struggle with mathematics**, even simple mathematics. Below, Claude incorrectly assesses the math problem as correctly solved, even though there's an obvious arithmetic mistake in the second step. Note that Claude actually catches the mistake when going through step-by-step, but doesn't jump to the conclusion that the overall solution is wrong.

Modify the `PROMPT` and / or the `SYSTEM_PROMPT` to make Claude grade the solution as `incorrectly` solved, rather than correctly solved. 


In [11]:
# System prompt - if you don't want to use a system prompt, you can leave this variable set to an empty string
SYSTEM_PROMPT = "Act like a  logic bot, that solves the prompt step by step"

# Prompt
PROMPT = """Is this equation solved correctly below?

2x - 3 = 9
2x = 6
x = 3"""

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    if "incorrect" in text or "not correct" in text.lower():
        return True
    else:
        return False

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

# Let me verify this step-by-step:

**Step 1:** 2x - 3 = 9
- Add 3 to both sides: 2x = 9 + 3
- Result: 2x = 12 ✓

**Step 2:** 2x = 6
- ❌ **This is incorrect.** It should be 2x = 12, not 2x = 6

**Step 3:** x = 3
- ❌ **This is incorrect as a result.** If we divide 2x = 12 by 2, we get x = 6

## Correct Solution:
- 2x - 3 = 9
- 2x = 12
- **x = 6**

## Verification:
2(6) - 3 = 12 - 3 = 9 ✓

**No, the equation was not solved correctly.** There was an arithmetic error in the first step where 9 + 3 was incorrectly calculated as 6 instead of 12.

--------------------------- GRADING ---------------------------
This exercise has been correctly solved: True


### Congrats!

If you've solved all exercises up until this point, you're ready to move to the next chapter. Happy prompting!

---

## Example Playground

This is an area for you to experiment freely with the prompt examples shown in this lesson and tweak prompts to see how it may affect Claude's responses.

In [ ]:
# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a cat."

# Prompt
PROMPT = "In one sentence, what do you think about skateboarding?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

In [ ]:
# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a logic bot designed to answer complex logic problems."

# Prompt
PROMPT = "Jack is looking at Anne. Anne is looking at George. Jack is married, George is not, and we don’t know if Anne is married. Is a married person looking at an unmarried person?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))